# AmsiNet - Step 25: AmsiNet-E on MDIW-13, zero-shot, with context (A100, about 40-50 minutes)
**Why.** The paper's cross-dataset result uses the older EfficientViT-B0 word agent (AmsiNet-T). The paper's headline system is AmsiNet-E (EfficientNet-B0). Step 16 did not save model weights, so this notebook (1) retrains the EfficientNet-B0 word agent on BSTD with the **same recipe and seeds** and saves the weights to `agents/effb0w_seed{k}.pt`, (2) applies it **without any MDIW-13 training** to all 37,311 printed MDIW-13 words, and (3) adds the free leave-one-out context of the line and of the document (unit weights, nothing tuned on MDIW-13), as in the earlier T analysis.

Needs everything from Step 2 (crops, `crops_meta.csv`, `bstd_index.json`), and `data/MultiscriptDatabase.zip` plus `mdiw/printed_words.csv` from Step 4. Prints a table to paste back.

In [ ]:
# 1. Setup
from google.colab import drive
import os, subprocess, sys
if not os.path.ismount('/content/drive'): drive.mount('/content/drive')
assert os.path.exists('/content/drive/MyDrive/amsinet/data/bstd_index.json'), 'Drive is not mounted or amsinet/data is missing: reconnect and run this cell again'
try: import timm
except ImportError: subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'timm'], check=True); import timm
import json, glob, random, time, math, numpy as np, pandas as pd, torch, torch.nn as nn
from PIL import Image
from scipy import stats
from scipy.special import softmax
from sklearn.metrics import f1_score
ROOT, LOCAL = '/content/drive/MyDrive/amsinet', '/content/bstd'; os.makedirs(LOCAL, exist_ok=True)
CFG = dict(SEEDS=[0, 1, 2, 3, 4], EPOCHS=8, BATCH=64, WORKERS=8)
D = json.load(open(f'{ROOT}/data/bstd_index.json', encoding='utf-8')); SCRIPTS, INDEX = D['scripts'], D['index']; IDX = {r['id']: r for r in INDEX}
META = pd.read_csv(f'{ROOT}/data/crops_meta.csv'); SIDX = {s: i for i, s in enumerate(SCRIPTS)}; C = len(SCRIPTS); META['y'] = META.script.map(SIDX).astype(int)
print(C, 'scripts:', SCRIPTS); print(META.groupby('split').size().to_dict())
CROPS, CROPS_TAR = f'{LOCAL}/crops224', f'{ROOT}/data/crops224.tar'
if not os.path.isdir(CROPS) and os.path.exists(CROPS_TAR): print('restoring crops'); subprocess.run(['tar', '-xf', CROPS_TAR, '-C', LOCAL], check=True)
assert os.path.exists(META.path.iloc[0]), 'crops not found on disk: ' + META.path.iloc[0]
print('torch', torch.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
# MDIW-13: unzip once, keep the same local paths as Step 4
import glob
MD_LOCAL, MD_ZIP = '/content/mdiw', f'{ROOT}/data/MultiscriptDatabase.zip'
if not glob.glob(f'{MD_LOCAL}/**/*.png', recursive=True): print('unzipping MDIW-13'); subprocess.run(['unzip', '-q', '-o', MD_ZIP, '-d', MD_LOCAL], check=True)
WORDS = pd.read_csv(f'{ROOT}/mdiw/printed_words.csv'); assert os.path.exists(WORDS.path.iloc[0]), 'MDIW-13 word images not found at ' + WORDS.path.iloc[0]
print('MDIW-13 printed words:', len(WORDS), WORDS.split.value_counts().to_dict())

## 2. Training utilities (the Step 16 code, now saving the weights)

In [ ]:
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
DEV = 'cuda'; FILL = (124, 116, 104)
class Crops(Dataset):
    def __init__(self, paths, labels, tf): self.p, self.y, self.tf = list(paths), list(labels), tf
    def __len__(self): return len(self.p)
    def __getitem__(self, i): return self.tf(Image.open(self.p[i]).convert('RGB')), self.y[i]
def tfs(model):
    cfg = timm.data.resolve_data_config({}, model=model); norm = T.Normalize(cfg['mean'], cfg['std'])
    return (T.Compose([T.RandomAffine(5, translate=(0.03, 0.03), scale=(0.9, 1.1), fill=FILL), T.ColorJitter(0.3, 0.3, 0.2), T.ToTensor(), norm]), T.Compose([T.ToTensor(), norm]))
def set_seed(s): random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
@torch.no_grad()
def predict(model, df, tf):
    model.eval(); out = []
    for x, _ in DataLoader(Crops(df.path, df.y, tf), batch_size=256, num_workers=CFG['WORKERS']):
        with torch.autocast('cuda', dtype=torch.float16): out.append(model(x.to(DEV)).float().softmax(1).cpu())
    return torch.cat(out).numpy()
def mf1(P, y): return float(f1_score(y, P.argmax(1), average='macro', labels=list(range(C)), zero_division=0))
def train(tag, model_name, tr, va, epochs, lr, seed):
    # trains on dataframe tr, selects the epoch by macro-F1 on dataframe va, saves probabilities for ALL val and test words
    if os.path.exists(f'{ROOT}/agents/{tag}_seed{seed}.pt'): print('skip (done):', tag, seed); return
    set_seed(seed); model = timm.create_model(model_name, pretrained=True, num_classes=C).to(DEV); ttf, etf = tfs(model)
    L = DataLoader(Crops(tr.path, tr.y, ttf), batch_size=CFG['BATCH'], shuffle=True, num_workers=CFG['WORKERS'], pin_memory=True, drop_last=True)
    crit = nn.CrossEntropyLoss(label_smoothing=0.1); opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.05)
    total = epochs * len(L); warm = max(1, int(0.05 * total)); sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
    scaler = torch.amp.GradScaler('cuda'); best, state = -1, None
    for ep in range(epochs):
        model.train(); t0 = time.time(); rl = 0
        for x, y in L:
            x, y = x.to(DEV, non_blocking=True), torch.as_tensor(y).to(DEV)
            with torch.autocast('cuda', dtype=torch.float16): loss = crit(model(x), y)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(opt); scaler.update(); sched.step(); rl += loss.item()
        Pv = predict(model, va, etf); f = mf1(Pv, va.y.values)
        print(f'{tag} seed{seed} ep{ep + 1}: loss {rl / len(L):.4f} val acc {(Pv.argmax(1) == va.y.values).mean():.4f} macro-F1 {f:.4f} ({time.time() - t0:.0f}s)')
        if f > best: best = f; state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(state); torch.save(state, f'{ROOT}/agents/{tag}_seed{seed}.pt')
    for sp in ('val', 'test'):
        d = META[META.split == sp]; P = predict(model, d, etf); np.save(f'{ROOT}/agents/{tag}_seed{seed}_{sp}_probs.npy', P.astype(np.float16)); pd.DataFrame({'path': d.path.values}).to_csv(f'{ROOT}/agents/{tag}_seed{seed}_{sp}_paths.csv', index=False)
    print(f'done {tag} seed{seed}: best val macro-F1 {best:.4f}'); del model; torch.cuda.empty_cache()

## 3. Train the EfficientNet-B0 word agent (five seeds, same recipe as Step 16)

In [ ]:
TRN, VAL = META[META.split == 'train'], META[META.split == 'val']
for seed in CFG['SEEDS']: train('effb0w', 'efficientnet_b0.ra_in1k', TRN, VAL, CFG['EPOCHS'], 5e-4, seed)
TE = META[META.split == 'test']; yt = TE.y.values
for seed in CFG['SEEDS']:
    P = np.load(f'{ROOT}/agents/effb0w_seed{seed}_test_probs.npy').astype(np.float32); print(f'BSTD test, seed {seed}: accuracy {100 * (P.argmax(1) == yt).mean():.2f} macro-F1 {100 * mf1(P, yt):.2f}')
print('(Step 16 gave 91.89 / 85.63 on average; small differences are normal.)')

## 4. Zero-shot inference on the MDIW-13 printed words

In [ ]:
FILL = (124, 116, 104)
def letterbox(im, w, h):
    iw, ih = im.size; s = min(w / iw, h / ih); nw, nh = max(1, int(round(iw * s))), max(1, int(round(ih * s)))
    im = im.resize((nw, nh), Image.BICUBIC, reducing_gap=3.0); cv = Image.new('RGB', (w, h), FILL); cv.paste(im, ((w - nw) // 2, (h - nh) // 2)); return cv
mcfg = timm.data.resolve_data_config({}, model=timm.create_model('efficientnet_b0.ra_in1k', pretrained=False, num_classes=C)); tfm = T.Compose([T.ToTensor(), T.Normalize(mcfg['mean'], mcfg['std'])])
class WDS(Dataset):
    def __init__(s, paths): s.p = list(paths)
    def __len__(s): return len(s.p)
    def __getitem__(s, i): return tfm(letterbox(Image.open(s.p[i]).convert('RGB'), 224, 224))
@torch.no_grad()
def predict_paths(model, paths):
    model.eval(); out = []
    for x in DataLoader(WDS(paths), batch_size=256, num_workers=CFG['WORKERS']):
        with torch.autocast('cuda', dtype=torch.float16): out.append(model(x.to(DEV)).float().softmax(1).cpu())
    return torch.cat(out).numpy()
for seed in CFG['SEEDS']:
    f = f'{ROOT}/mdiw/zeroshot_effb0w_seed{seed}_printed_probs.npy'
    if os.path.exists(f): print('skip', seed); continue
    m = timm.create_model('efficientnet_b0.ra_in1k', pretrained=False, num_classes=C).to(DEV); m.load_state_dict(torch.load(f'{ROOT}/agents/effb0w_seed{seed}.pt', map_location=DEV))
    np.save(f, predict_paths(m, WORDS.path).astype(np.float32)); print('saved', f); del m; torch.cuda.empty_cache()

## 5. Free context on MDIW-13 (leave-one-out, unit weights, nothing tuned on MDIW-13)

In [ ]:
EPS = 1e-4; ym = WORDS.script.map(SIDX).values if 'SIDX' in globals() else WORDS.script.map({s: i for i, s in enumerate(SCRIPTS)}).values
def loo(lg, key):
    k = pd.factorize(WORDS[key])[0]; Sg = np.zeros((k.max() + 1, C)); n = np.zeros(k.max() + 1); np.add.at(Sg, k, lg); np.add.at(n, k, 1)
    o = n[k] - 1; Fx = np.zeros_like(lg); m = o > 0; Fx[m] = (Sg[k][m] - lg[m]) / o[m][:, None]; return Fx
def met(y, p): return 100 * (p == y).mean(), 100 * f1_score(y, p, average='macro', labels=list(range(C)), zero_division=0)
te = (WORDS.split == 'test').values; rows = []
for seed in CFG['SEEDS']:
    lg = np.log(np.clip(np.load(f'{ROOT}/mdiw/zeroshot_effb0w_seed{seed}_printed_probs.npy'), EPS, 1)); Fl, Fd = loo(lg, 'line'), loo(lg, 'doc')
    for name, s in [('word agent alone', lg), ('+ line context', lg + Fl), ('+ line and document context', lg + Fl + Fd)]:
        for sp, sel in (('all words', np.ones(len(WORDS), bool)), ('test documents', te)):
            a, f = met(ym[sel], s[sel].argmax(1)); rows.append((name, sp, seed, a, f))
R = pd.DataFrame(rows, columns=['system', 'words', 'seed', 'acc', 'f1'])
tci = lambda v: stats.t.ppf(0.975, len(v) - 1) * np.std(v, ddof=1) / np.sqrt(len(v))
Q = R.groupby(['system', 'words'], sort=False).agg(acc=('acc', 'mean'), acc_ci=('acc', tci), f1=('f1', 'mean'), f1_ci=('f1', tci)).round(2); display(Q)
Q.to_csv(f'{ROOT}/results/mdiw_effb0w_context.csv')
print('Per-script F1 (all words, mean of seeds): alone vs + line and document context')
PS = {}
for seed in CFG['SEEDS']:
    lg = np.log(np.clip(np.load(f'{ROOT}/mdiw/zeroshot_effb0w_seed{seed}_printed_probs.npy'), EPS, 1)); s2 = lg + loo(lg, 'line') + loo(lg, 'doc')
    PS[seed] = (100 * f1_score(ym, lg.argmax(1), average=None, labels=list(range(C))), 100 * f1_score(ym, s2.argmax(1), average=None, labels=list(range(C))))
display(pd.DataFrame({'script': SCRIPTS, 'alone': np.mean([v[0] for v in PS.values()], 0).round(1), '+ line and document context': np.mean([v[1] for v in PS.values()], 0).round(1)}))
print('Every MDIW-13 line and document holds one script, so the context gain is an upper bound that tests consistency, not mixed-script routing.')